# <center> Predicting Zebrafish Movements in Virtual Reality

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from IPython.display import display
import functions
from tqdm import tqdm
from matplotlib import animation
from IPython.display import Image
import pickle


## Context

This project is part of a collaboration between EPFL and UNIL.
It aims to understand the perceptual and computational rules underlying social interactions in zebrafish through behavioral and neural data analysis.

In this experiment, a single zebrafish swims in a controlled environment equipped with a virtual reality system.  
One or several visual stimuli (dots) are projected on a screen, simulating the motion patterns of another fish, typically a sequence of “bursts and glides” characteristic of zebrafish swimming behavior.  
The fish’s trajectory, orientation, and velocity are continuously tracked, as well as the trajectories and properties of the stimuli.

## Objective :

The goal of this notebook is to compare two types of data representation and to select one of them to continue further preprocessing. The fish data representations are in

> 

>


 To decide which one is better, we will run an XGBoost using the two representations and compare the RMSE. In order to be 'fair' between the two representation, we do the same preprocessing and feature engineering for them.

In [8]:
with open("polar_dict.pkl", "rb") as f:
    polar_dict = pickle.load(f)


df_polar = functions.stack_dict_pipeline(polar_dict)

In [ ]:
#keep only 500000 first rows for faster processing
df_polar = df_polar.iloc[:500000, :]

Adding trial onset features. Will be usefull to check if the polar pixel encoding is correct.

## <center> Exploring dataset and preprocessing

In [33]:
print("Polar dataframe shape : " + str(df_polar.shape))

Polar dataframe shape : (500000, 31)


In [34]:
df_polar.head()

,frame,source_frame,stimulus,f_x,f_y,f_ori,d0_stim,d1_stim,d2_stim,d3_stim,...,d3_radius,d4_angle,d4_radius,d5_angle,d5_radius,d6_angle,d6_radius,d7_angle,d7_radius,fish_id
0,0,0,inter_stim_pause,NaN,NaN,NaN,inter_stim_pause,inter_stim_pause,inter_stim_pause,inter_stim_pause,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,1,inter_stim_pause,NaN,NaN,NaN,inter_stim_pause,inter_stim_pause,inter_stim_pause,inter_stim_pause,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2,2,inter_stim_pause,NaN,NaN,NaN,inter_stim_pause,inter_stim_pause,inter_stim_pause,inter_stim_pause,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,3,3,inter_stim_pause,NaN,NaN,NaN,inter_stim_pause,inter_stim_pause,inter_stim_pause,inter_stim_pause,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,4,4,inter_stim_pause,NaN,NaN,NaN,inter_stim_pause,inter_stim_pause,inter_stim_pause,inter_stim_pause,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [35]:
df_polar.dtypes.value_counts()

float64    19
object      9
int64       3
Name: count, dtype: int64

In [36]:
cat_cols = df_polar.select_dtypes(include=['object']).columns
cat_cols.tolist()

['stimulus',
 'd0_stim',
 'd1_stim',
 'd2_stim',
 'd3_stim',
 'd4_stim',
 'd5_stim',
 'd6_stim',
 'd7_stim']

In [37]:
unique_conditions = df_polar['stimulus'].unique()
print(f"There are {len(unique_conditions)} unique conditions:\n")
for c in unique_conditions:
    print("-", c)

There are 17 unique conditions:

- inter_stim_pause
- CLsemi1800_g4-0_s3.5-3.5
- CLsemi1800_g0-2_s3.5-3.5
- CLsemi1800_g0-4_s3.5-3.5
- CLsemi1800_g8-0_s3.5-3.5
- CLsemi1800_g0-4_s0-0
- CLsemi1800_g8-0_s0-0
- CLsemi1800_g6-0_s3.5-3.5
- CLsemi1800_g0-8_s0-0
- CLsemi1800_g1-0_s3.5-3.5
- CLsemi1800_g0-1_s3.5-3.5
- CLsemi1800_g2-0_s3.5-3.5
- CLsemi1800_g0-8_s3.5-3.5
- CLsemi1800_g0-6_s3.5-3.5
- CLsemi1800_g4-0_s0-0
- CLsemi1800_g0-1_s0-0
- CLsemi1800_g1-0_s0-0


##### There is a non sens to give conditional experiment information to the model. We want to give the present and past full state position of the fish and its visual field.

In [38]:
df_polar = df_polar.drop(columns=['stimulus'])

# Transform dk_stim for each stimulus k into 0 if the feature is 'none' else 1
for k in range(8):
    df_polar[f'd{k}_stim'] = df_polar[f'd{k}_stim'].apply(lambda x: 0 if x == 'none' else 1)
df_polar = df_polar.drop(columns=['fish_id']) # Make no sens to keep fish_id for prediction
df_polar = df_polar.drop(columns=['frame']) # Make no sens to keep frame for prediction

In [39]:
print("------------ Polar dataframe ------------")
df_polar, _ = functions.remove_nan_plus_constant_and_correlated_features(df_polar, corr_threshold=1.0)

------------ Polar dataframe ------------
The dataset contains : 0.6% of NaN.
There are few NaN, therefore we make the choice to remove them.
🧩 Constant features removed: ['d0_stim']
✅ No highly correlated features found.


In [40]:
df_polar = functions.compute_d_orientation(df_polar)
df_polar = functions.compute_speed(df_polar)

df_polar = df_polar.drop(columns=['f_x', 'f_y', 'f_ori'])  # drop original position columns after computing speed

df_polar.head()

,f_speed,d_ori,source_frame,d1_stim,d2_stim,d3_stim,d4_stim,d5_stim,d6_stim,d7_stim,...,d3_angle,d3_radius,d4_angle,d4_radius,d5_angle,d5_radius,d6_angle,d6_radius,d7_angle,d7_radius
0,NaN,NaN,0,1,1,1,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,1,1,1,1,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,2,1,1,1,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,3,1,1,1,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,4,1,1,1,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
df_polar_feature_engineered = functions.add_engineered_features(df_polar, representation="polar", n_stimuli=8)

In [42]:
df_polar_feature_engineered = df_polar_feature_engineered.drop(columns=[f"d{k}_{attr}" for k in range(8) for attr in ["stim", "angle", "radius"] if f"d{k}_{attr}" in df_polar_feature_engineered.columns])

In [43]:
df_polar_feature_engineered.head()

,f_speed,d_ori,source_frame,d1_dist,d1_side,d2_dist,d2_side,d3_dist,d3_side,d4_dist,...,min_dist_active,mean_dist_active,mean_angle_active,lateral_balance,f_speed_sq,f_speed_rolling_mean_30,f_speed_trend_30,d_ori_abs,d_ori_rolling_mean_30,d_ori_trend_30
0,NaN,NaN,0,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,-3.0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,1,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,-3.0,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,2,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,-3.0,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,3,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,-3.0,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,4,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,-3.0,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
df_polar_lag, created_lag_cols_names = functions.add_lag_features_for_all(df_polar_feature_engineered, 1, subsample=5)

Created 168 lag features for 28 base columns, using lags from 5 to 30 frames (step=5, horizon ≈ 1.00 s at 30 fps).


/Users/nayan/Desktop/Research/work with data/mp_semiCL_behavior/notebooks/functions.py:826: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag_col] = df[col].shift(lag)
/Users/nayan/Desktop/Research/work with data/mp_semiCL_behavior/notebooks/functions.py:826: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag_col] = df[col].shift(lag)
/Users/nayan/Desktop/Research/work with data/mp_semiCL_behavior/notebooks/functions.py:826: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.in

In [45]:
df_polar_lag.head()

,f_speed,d_ori,source_frame,d1_dist,d1_side,d2_dist,d2_side,d3_dist,d3_side,d4_dist,...,d_ori_rolling_mean_30_lag_15,d_ori_rolling_mean_30_lag_20,d_ori_rolling_mean_30_lag_25,d_ori_rolling_mean_30_lag_30,d_ori_trend_30_lag_5,d_ori_trend_30_lag_10,d_ori_trend_30_lag_15,d_ori_trend_30_lag_20,d_ori_trend_30_lag_25,d_ori_trend_30_lag_30
0,NaN,NaN,0,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,1,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,2,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,3,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,4,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Pixel Polar Encoding

In [46]:
r_bins=20
theta_bins=18

px_features = functions.build_polar_pixel_encoding_df(df_polar, r_bins, theta_bins, max_radius=150)

In [47]:
px_features.head()

,px_r0_t0,px_r0_t1,px_r0_t2,px_r0_t3,px_r0_t4,px_r0_t5,px_r0_t6,px_r0_t7,px_r0_t8,px_r0_t9,...,px_r19_t8,px_r19_t9,px_r19_t10,px_r19_t11,px_r19_t12,px_r19_t13,px_r19_t14,px_r19_t15,px_r19_t16,px_r19_t17
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [48]:
df = pd.concat([df_polar_lag, px_features], axis=1)

In [49]:
df.head()

,f_speed,d_ori,source_frame,d1_dist,d1_side,d2_dist,d2_side,d3_dist,d3_side,d4_dist,...,px_r19_t8,px_r19_t9,px_r19_t10,px_r19_t11,px_r19_t12,px_r19_t13,px_r19_t14,px_r19_t15,px_r19_t16,px_r19_t17
0,NaN,NaN,0,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,NaN,NaN,1,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,NaN,NaN,2,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,NaN,NaN,3,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,NaN,NaN,4,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [50]:
delta_t = 15

df['f_speed_next'] = df['f_speed'].shift(-delta_t)
df['d_ori_next'] = df['d_ori'].shift(-delta_t)

# Drop only rows where future targets are missing
df = df.dropna(subset=["f_speed_next", "d_ori_next"])

df.head()

,f_speed,d_ori,source_frame,d1_dist,d1_side,d2_dist,d2_side,d3_dist,d3_side,d4_dist,...,px_r19_t10,px_r19_t11,px_r19_t12,px_r19_t13,px_r19_t14,px_r19_t15,px_r19_t16,px_r19_t17,f_speed_next,d_ori_next
131,NaN,NaN,131,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.363
132,NaN,NaN,132,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005
133,NaN,NaN,133,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.017
134,NaN,NaN,134,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.083
135,NaN,NaN,135,NaN,-1.0,NaN,-1.0,NaN,-1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.105


## <center> Creates train - val - test set

##### Our goal is to predict the fish’s next position and orientation $x_{t+1}$, $y_{t+1}$, and $\text{ori}_{t+1}$ using only the information available at the current time $t$.  
##### Formally, this means estimating the conditional expectations:

$$
\begin{aligned}
\hat{x}_{t+1} &= \mathbb{E}[x_{t+1} \mid \mathcal{F}_t] \\
\hat{y}_{t+1} &= \mathbb{E}[y_{t+1} \mid \mathcal{F}_t] \\
\hat{\text{ori}}_{t+1} &= \mathbb{E}[\text{ori}_{t+1} \mid \mathcal{F}_t]
\end{aligned}
$$

##### where $\mathcal{F}_t$ represents all information known at time $t$ (fish position, stimuli, condition, etc.).  
##### In other words, the model learns the most likely future state of the fish given the current context.

In [51]:
target_cols = ["f_speed_next", "d_ori_next"]

X_train, X_val, X_test, target_speed_train, target_speed_val, target_speed_test, target_ori_train, target_ori_val, target_ori_test = functions.temporal_train_val_test_split(df, target_cols)

✅ Temporal split complete:
  Train: 349813 rows (70.0%)
  Val:   74960 rows (15.0%)
  Test:  74960 rows (15.0%)


##### Filtering the test set

Inter-stimulus pause frames (where all d*_stim flags are 0) contain no stimulus information. As our goal is to make a choici between 2 representations of visual input there is no sens to keep these rows in the test set.
The training set remains unfiltered, so the model still learns baseline fish dynamics.

In [52]:
# Identify all stimulus flag columns in X_test
stim_flag_cols = [col for col in X_test.columns 
                  if col.startswith("d") and col.endswith("_stim")]

# Create mask: True = at least one active stimulus in that row
active_stim_mask = (X_test[stim_flag_cols].sum(axis=1) > 0)

# Apply mask to X_test + targets
X_test_filtered = X_test[active_stim_mask].copy()
target_speed_test_filtered = target_speed_test[active_stim_mask].copy()
target_ori_test_filtered = target_ori_test[active_stim_mask].copy()

print(f"Original test size: {len(X_test)} rows")
print(f"Filtered test size: {len(X_test_filtered)} rows")
print(f"Removed {len(X_test) - len(X_test_filtered)} inter_stim_pause rows")

Original test size: 74960 rows
Filtered test size: 0 rows
Removed 74960 inter_stim_pause rows


## <center> Train XGBoost

In [53]:
cat_cols = X_train.select_dtypes(include=['object']).columns
cat_cols.tolist()

[]

In [54]:
# --- Train model for speed prediction ---
model_speed = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model_speed.fit(X_train, target_speed_train, eval_set=[(X_val, target_speed_val)], verbose=False)


# --- Train model for Orientation prediction ---
model_ori = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model_ori.fit(X_train, target_ori_train, eval_set=[(X_val, target_ori_val)], verbose=False)

# --- Predictions ---
pred_speed = model_speed.predict(X_test)
pred_ori = model_ori.predict(X_test)

# --- RMSE (Root Mean Squared Error) ---
rmse_speed = mean_squared_error(target_speed_test, pred_speed, squared=False)
rmse_ori = mean_squared_error(target_ori_test, pred_ori, squared=False)


# --- Display results ---
print("📊 Model Performance Summary:")
print(f"RMSE Speed: {rmse_speed:.4f}")
print(f"RMSE Orientation: {rmse_ori:.4f}")

📊 Model Performance Summary:
RMSE Speed: 0.7925
RMSE Orientation: 0.9197


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [55]:
rmse_speed_norm = rmse_speed / (df['f_speed'].max() - df['f_speed'].min())
rmse_ori_norm = rmse_ori / (df['d_ori'].max() - df['d_ori'].min())

In [56]:
from sklearn.metrics import mean_squared_error
import pandas as pd

# --- Static baseline ---
# Hypothesis: fish does not move (same position and orientation)
baseline_pred_speed = df.loc[X_test.index, 'f_speed']
baseline_pred_ori = df.loc[X_test.index, 'd_ori']

baseline_rmse_speed = mean_squared_error(target_speed_test, baseline_pred_speed, squared=False) / (df['f_speed'].max() - df['f_speed'].min())
baseline_rmse_ori = mean_squared_error(target_ori_test, baseline_pred_ori, squared=False) / (df['d_ori'].max() - df['d_ori'].min())

/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [57]:
# Build summary table
results_df = pd.DataFrame({
    "Metric": ["Speed", "Change in Orientation (rad)"],
    "Model RMSE": [rmse_speed_norm, rmse_ori_norm],
    "Static Baseline RMSE": [baseline_rmse_speed, baseline_rmse_ori],
})

# Format for readability
results_df = results_df.round(4)

# Display nicely
print("📊 Model Performance Summary:\n")
display(
    results_df.style.set_table_styles([
        {"selector": "th", "props": [("font-weight", "bold"), ("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "center")]}
    ])
    .set_caption("Comparison of Model and Baselines (normalized RMSE)")
)

📊 Model Performance Summary:



,Metric,Model RMSE,Static Baseline RMSE
0,Speed,0.027700,0.039400
1,Change in Orientation (rad),0.073200,0.102400
